# SVM lineal sobre la muestra inicial de Lending Club

Este notebook agrega la comparaci?n que faltaba: **SVM lineal (`LinearSVC`) entrenado sobre la misma muestra inicial de 220,003 pr?stamos** usada en los experimentos tempranos. Se comparan regresi?n log?stica, SVM lineal, ?rbol de decisi?n e HistGradientBoosting con la misma preparaci?n, observaciones y particiones temporales.

La muestra usa entrenamiento 2007?2014, validaci?n 2015?2016 y prueba 2017?2018. La selecci?n de umbral se hace ?nicamente sobre validaci?n. `LinearSVC` no produce probabilidades; por eso ROC-AUC y AP usan su `decision_function`, mientras que las m?tricas de clasificaci?n usan un umbral aprendido en validaci?n. Estos resultados no deben mezclarse con el pipeline final entrenado sobre todos los pr?stamos elegibles.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, accuracy_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import features as ft
from src.config import (ANIO_CORTE_PRUEBA, ANIO_CORTE_VALIDACION,
                        SEMILLA, TAMANO_MUESTRA, localizar_datos)
from src.data import dividir_por_anio, escanear_y_muestrear
from src.experimentos import preparar_experimentos
from src.evaluate import elegir_umbral

RUTA_DATOS = None
TAMANO_OBJETIVO = TAMANO_MUESTRA
SALIDA = RAIZ / "outputs" / "svm_muestra"
SALIDA.mkdir(parents=True, exist_ok=True)
print(f"Semilla: {SEMILLA}; muestra objetivo: {TAMANO_OBJETIVO:,}")

## 1. Carga y verificaci?n de la muestra

`escanear_y_muestrear` recorre el archivo por bloques, conserva ?nicamente `Fully Paid` y `Charged Off`, y aplica el muestreo proporcional reproducible del proyecto. No se descarga ni modifica el dataset.

In [ ]:
ruta = localizar_datos(RUTA_DATOS)
muestra, auditoria = escanear_y_muestrear(
    ruta, tamano_muestra=TAMANO_OBJETIVO, semilla=SEMILLA, verbose=True
)
assert len(muestra) == 220003, f"Muestra inesperada: {len(muestra):,}"
assert set(muestra["default"].unique()) == {0, 1}
print(f"Archivo: {ruta}")
print(f"Filas fuente: {auditoria['total_filas']:,}")
print(f"Elegibles: {auditoria['total_elegibles']:,}")
print(f"Muestra verificada: {len(muestra):,}")
print(f"Prevalencia: {muestra['default'].mean():.4f}")

In [ ]:
entrenamiento, validacion, prueba = dividir_por_anio(
    muestra, ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA
)
particiones = {
    "entrenamiento": entrenamiento,
    "validacion": validacion,
    "prueba": prueba,
}
resumen_particiones = pd.DataFrame([
    {
        "conjunto": nombre,
        "filas": len(tabla),
        "anio_min": int(tabla["anio"].min()),
        "anio_max": int(tabla["anio"].max()),
        "default_pct": 100 * tabla["default"].mean(),
    }
    for nombre, tabla in particiones.items()
])
display(resumen_particiones)

## 2. Preparaci?n sin fuga

Se utiliza el escenario sin indicadores ni variables de precio de Lending Club, igual que la referencia log?stica inicial. Las medianas, categor?as, escalas y columnas retenidas se ajustan solo con entrenamiento. La matriz transformada es compartida por los cuatro modelos para que la comparaci?n sea justa.

In [ ]:
faltantes_globales = auditoria["faltantes_globales"]
columnas_vacias = faltantes_globales[
    faltantes_globales == auditoria["total_filas"]
].index.tolist()
preparados = preparar_experimentos(particiones, columnas_vacias)
datos = preparados["logistica_sin_indicadores"]
X = datos["X"]
y = {nombre: valores.to_numpy(dtype="int8")
     for nombre, valores in datos["y"].items()}
print(datos["variables"])
print("Columnas transformadas:", X["entrenamiento"].shape[1])
assert all(np.isfinite(tabla.to_numpy()).all() for tabla in X.values())

## 3. Modelos y entrenamiento

Los par?metros del SVM son los seleccionados en la b?squeda temporal del proyecto. El ?rbol y el boosting usan configuraciones fijas de referencia del experimento inicial. Para el SVM se conserva el margen de `decision_function`; no se interpreta como probabilidad.

In [ ]:
modelos = {
    "regresion_logistica": LogisticRegression(
        C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA
    ),
    "svm_lineal": LinearSVC(
        C=0.010053940194693016, class_weight="balanced",
        dual="auto", max_iter=7000, random_state=SEMILLA
    ),
    "arbol_decision": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=1000, random_state=SEMILLA
    ),
    "gradient_boosting": HistGradientBoostingClassifier(
        max_iter=75, max_leaf_nodes=15, learning_rate=0.08,
        l2_regularization=1.0, early_stopping=False, random_state=SEMILLA
    ),
}

def puntuaciones(modelo, X_bloque):
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(X_bloque)[:, 1]
    return modelo.decision_function(X_bloque)

scores = {}
tiempos = {}
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X["entrenamiento"], y["entrenamiento"])
    tiempos[nombre] = time.perf_counter() - inicio
    scores[nombre] = {conjunto: puntuaciones(modelo, X[conjunto])
                      for conjunto in particiones}
    print(f"{nombre}: {tiempos[nombre]:.2f} s")

## 4. M?tricas

ROC-AUC y AP miden ranking y son comparables para todos los modelos. Para recall, precisi?n, F1 y accuracy se elige el umbral que maximiza F1 en validaci?n y se congela para prueba. No se calcula Brier para el margen SVM sin calibrar.

In [ ]:
def metricas_modelo(nombre, scores_modelo):
    umbral = elegir_umbral(y["validacion"], scores_modelo["validacion"])
    filas = []
    for conjunto in particiones:
        objetivo = y[conjunto]
        score = scores_modelo[conjunto]
        pred = (score >= umbral).astype("int8")
        vn, fp, fn, vp = confusion_matrix(objetivo, pred, labels=[0, 1]).ravel()
        filas.append({
            "modelo": nombre, "conjunto": conjunto, "n": len(objetivo),
            "ROC_AUC": roc_auc_score(objetivo, score),
            "AP": average_precision_score(objetivo, score),
            "umbral_validacion": umbral,
            "precision": precision_score(objetivo, pred, zero_division=0),
            "recall": recall_score(objetivo, pred, zero_division=0),
            "F1": f1_score(objetivo, pred, zero_division=0),
            "accuracy": accuracy_score(objetivo, pred),
            "VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "segundos_entrenamiento": tiempos[nombre],
        })
    return filas

metricas = pd.DataFrame([
    fila for nombre, score in scores.items()
    for fila in metricas_modelo(nombre, score)
])
display(metricas.round(4))
metricas.to_csv(SALIDA / "metricas_svm_muestra.csv", index=False)

In [ ]:
resumen_test = (metricas[metricas["conjunto"] == "prueba"]
                .sort_values("ROC_AUC", ascending=False))
display(resumen_test.round(4))

## 5. Curvas ROC y Precision?Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for nombre, score in scores.items():
    fpr, tpr, _ = roc_curve(y["prueba"], score["prueba"])
    precision, recall, _ = precision_recall_curve(y["prueba"], score["prueba"])
    auc = roc_auc_score(y["prueba"], score["prueba"])
    ap = average_precision_score(y["prueba"], score["prueba"])
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC={auc:.4f})")
    axes[1].plot(recall, precision, label=f"{nombre} (AP={ap:.4f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="Tasa de falsos positivos", ylabel="Recall", title="ROC en prueba")
axes[1].set(xlabel="Recall", ylabel="Precisi?n", title="Precision?Recall en prueba")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(SALIDA / "curvas_svm_muestra.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Conclusiones y alcance

- El SVM lineal queda comparado sobre exactamente la misma muestra y particiones que los dem?s modelos de esta etapa.
- Su `decision_function` sirve para ROC-AUC y AP, pero no es una probabilidad calibrada.
- El umbral se aprende en validaci?n y no se reajusta en prueba.
- Estos resultados corresponden a la muestra inicial de 220,003 pr?stamos y no sustituyen los resultados del pipeline final entrenado con todos los pr?stamos elegibles.
- La muestra inicial es ?til para comparar familias y costo de c?mputo; la decisi?n final debe basarse en el flujo temporal completo, calibrado y documentado en los notebooks 08?10.

In [ ]:
(SALIDA / "alcance.md").write_text(
    "Resultados sobre la muestra inicial verificada de 220,003 pr?stamos. "
    "No mezclar con el pipeline final completo; LinearSVC usa decision_function "
    "y no probabilidades calibradas en este notebook.\n",
    encoding="utf-8",
)
print(f"Resultados guardados en: {SALIDA.resolve()}")